# SSD Object Detection: VGG16 vs MobileNetV2 Backbone
**Pascal VOC2007 · Google Colab T4**

Steps:
1. Clone repo & install dependencies
2. Download VOC2007 dataset
3. Download pretrained weights
4. Train MobileNetV2-SSD
5. Evaluate mAP & compare speed/params

In [5]:
!pip install torch
# Verify GPU is available
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

  Using cached filelock-3.29.0-py3-none-any.whl.metadata (2.0 kB)
  Using cached typing_extensions-4.15.0-py3-none-any.whl.metadata (3.3 kB)
  Using cached setuptools-81.0.0-py3-none-any.whl.metadata (6.6 kB)
  Using cached sympy-1.14.0-py3-none-any.whl.metadata (12 kB)
  Using cached networkx-3.6.1-py3-none-any.whl.metadata (6.8 kB)
  Using cached jinja2-3.1.6-py3-none-any.whl.metadata (2.9 kB)
  Using cached mpmath-1.3.0-py3-none-any.whl.metadata (8.6 kB)
  Using cached markupsafe-3.0.3-cp313-cp313-macosx_11_0_arm64.whl.metadata (2.7 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.0/88.0 MB 10.6 MB/s  0:00:08m0:00:0100:01
Using cached setuptools-81.0.0-py3-none-any.whl (1.1 MB)
Using cached networkx-3.6.1-py3-none-any.whl (2.1 MB)
Using cached sympy-1.14.0-py3-none-any.whl (6.3 MB)
Using cached mpmath-1.3.0-py3-none-any.whl (536 kB)
Using cached typing_extensions-4.15.0-py3-none-any.whl (44 kB)
Using cached filelock-3.29.0-py3-none-any.whl (39 kB)
Using cached jinja2-3.1.6-py3-non

/Users/rishichudasama/Documents/CV_Final/.venv/lib/python3.13/site-packages/torch/_subclasses/functional_tensor.py:362: UserWarning: Failed to initialize NumPy: No module named 'numpy' (Triggered internally at /Users/runner/work/pytorch/pytorch/torch/csrc/utils/tensor_numpy.cpp:84.)
  cpu = _conversion_method_template(device=torch.device("cpu"))


CUDA available: False


## 1. Clone Repo & Install Dependencies

Clones the project repo (which includes the fixed `mobilenet_ssd.py`, `train_mobilenet.py`, `eval_mobilenet.py`, and the updated `data/config.py`).

In [6]:
!git clone https://github.com/hashemalo/ssd-voc2007.git
%cd ssd-voc2007/ssd.pytorch
!pip install -q torch torchvision opencv-python matplotlib numpy tqdm

Cloning into 'ssd-voc2007'...
remote: Enumerating objects: 78, done.
remote: Counting objects: 100% (78/78), done.
remote: Compressing objects: 100% (57/57), done.
remote: Total 78 (delta 22), reused 74 (delta 18), pack-reused 0 (from 0)
Receiving objects: 100% (78/78), 3.84 MiB | 6.74 MiB/s, done.
Resolving deltas: 100% (22/22), done.
/Users/rishichudasama/Documents/CV_Final/ssd-voc2007/ssd-voc2007/ssd.pytorch


## 2. Download Pascal VOC2007

Downloads trainval + test splits (~870 MB total) and extracts them.
Uses explicit `-O` and `-C` flags so paths don't depend on shell `cd` state.
The final `ls` confirms the expected directory structure exists before you proceed.

In [ ]:
!mkdir -p data
!wget --show-progress -O data/VOCtrainval_06-Nov-2007.tar \
    http://host.robots.ox.ac.uk/pascal/VOC/voc2007/VOCtrainval_06-Nov-2007.tar
!wget --show-progress -O data/VOCtest_06-Nov-2007.tar \
    http://host.robots.ox.ac.uk/pascal/VOC/voc2007/VOCtest_06-Nov-2007.tar
!tar xf data/VOCtrainval_06-Nov-2007.tar -C data/
!tar xf data/VOCtest_06-Nov-2007.tar -C data/
print('Verifying dataset structure:')
!ls data/VOCdevkit/VOC2007/

In [ ]:
# Verify the specific files the dataloader needs exist before training
import os
required = [
    'data/VOCdevkit/VOC2007/ImageSets/Main/trainval.txt',
    'data/VOCdevkit/VOC2007/ImageSets/Main/test.txt',
    'data/VOCdevkit/VOC2007/Annotations',
    'data/VOCdevkit/VOC2007/JPEGImages',
]
all_ok = True
for path in required:
    exists = os.path.exists(path)
    print(f"{'OK' if exists else 'MISSING'}  {path}")
    if not exists:
        all_ok = False
if all_ok:
    print('\nDataset ready.')
else:
    print('\nOne or more paths are missing — re-run the download cell above.')

## 3. Download Pretrained Weights

- `vgg16_reducedfc.pth` — VGG16 backbone init weights for training VGG16-SSD
- `ssd300_mAP_77.43_v2.pth` — fully trained VGG16-SSD checkpoint for the baseline mAP eval

MobileNetV2 backbone weights are downloaded automatically by torchvision when `build_mobilenet_ssd` is first called.

In [ ]:
!mkdir -p weights
!wget --show-progress -O weights/vgg16_reducedfc.pth \
    https://s3.amazonaws.com/amdegroot-models/vgg16_reducedfc.pth
!wget --show-progress -O weights/ssd300_mAP_77.43_v2.pth \
    https://s3.amazonaws.com/amdegroot-models/ssd300_mAP_77.43_v2.pth
print('Weights:')
!ls -lh weights/

## 4. Train MobileNetV2-SSD

The model uses 4 feature maps (19×19, 10×10, 5×5, 3×3) with a MobileNetV2 backbone.
Prior boxes are generated from the MobileNet-specific anchor config and returned by the
model on every forward pass so the loss function receives `(loc, cls, priors)`.

Checkpoints saved every 10 epochs to `weights/`. Reduce `BATCH_SIZE` to 8 if you hit OOM.

In [ ]:
import sys

# Clear stale bytecode
!find /content/ssd-voc2007/ssd.pytorch -name "*.pyc" -delete

# Purge only our project modules from sys.modules.
# Use prefix matching — substring matching ("utils" in mod) would also
# hit torch.utils / torch.utils.data and corrupt PyTorch's dispatch system.
_ours = ['mobilenet_ssd', 'layers', 'utils', 'data']
for _mod in list(sys.modules.keys()):
    if any(_mod == x or _mod.startswith(x + '.') for x in _ours):
        del sys.modules[_mod]

import torch
import torch.optim as optim
from torch.utils.data import DataLoader
from utils.augmentations import SSDAugmentation
from data import VOCDetection, detection_collate
from mobilenet_ssd import build_mobilenet_ssd
from layers.modules import MultiBoxLoss
import time, os

NUM_CLASSES = 21
BATCH_SIZE  = 16
LR          = 1e-3
EPOCHS      = 50
DEVICE      = 'cuda' if torch.cuda.is_available() else 'cpu'
VOC_ROOT    = './data/VOCdevkit'

print(f'Training on: {DEVICE}')

# image_sets explicitly VOC2007 only — default also loads VOC2012 which we didn't download
dataset = VOCDetection(root=VOC_ROOT, image_sets=[('2007', 'trainval')],
                      transform=SSDAugmentation(300, (104, 117, 123)))
loader  = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True,
                     collate_fn=detection_collate, num_workers=0)
print(f'Dataset size: {len(dataset)}')

net       = build_mobilenet_ssd('train', num_classes=NUM_CLASSES).to(DEVICE)
optimizer = optim.SGD(net.parameters(), lr=LR, momentum=0.9, weight_decay=5e-4)
scheduler = optim.lr_scheduler.MultiStepLR(optimizer, milestones=[30, 40], gamma=0.1)
criterion = MultiBoxLoss(NUM_CLASSES, 0.5, True, 0, True, 3, 0.5, False, DEVICE == 'cuda')

os.makedirs('weights', exist_ok=True)

net.train()
for epoch in range(EPOCHS):
    epoch_loss = 0
    t0 = time.time()
    for imgs, targets in loader:
        imgs    = imgs.to(DEVICE)
        targets = [t.to(DEVICE) for t in targets]
        out     = net(imgs)
        optimizer.zero_grad()
        loss_l, loss_c = criterion(out, targets)
        loss = loss_l + loss_c
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
    scheduler.step()
    print(f"Epoch {epoch+1:>2}/{EPOCHS} | Loss: {epoch_loss/len(loader):.4f} | {time.time()-t0:.1f}s")
    if (epoch + 1) % 10 == 0:
        path = f'weights/mobilenet_ssd_epoch{epoch+1}.pth'
        torch.save(net.state_dict(), path)
        print(f'  -> Saved {path}')

print('Training complete.')

## 5. Speed & Parameter Comparison

In [ ]:
import torch, time
from ssd import build_ssd
from mobilenet_ssd import build_mobilenet_ssd

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

def count_params(model):
    return sum(p.numel() for p in model.parameters()) / 1e6

def benchmark_speed(model, n=100):
    model.eval()
    dummy = torch.randn(1, 3, 300, 300).to(DEVICE)
    with torch.no_grad():
        for _ in range(10):          # warmup
            model(dummy)
        t0 = time.time()
        for _ in range(n):
            model(dummy)
    return (time.time() - t0) / n * 1000  # ms/image

vgg_net = build_ssd('test', 300, 21).to(DEVICE)
vgg_net.load_weights('weights/ssd300_mAP_77.43_v2.pth')

mob_net = build_mobilenet_ssd('test', 21).to(DEVICE)
mob_net.load_state_dict(torch.load('weights/mobilenet_ssd_epoch50.pth', map_location=DEVICE))

print('=== Model Comparison ===')
print(f'VGG16-SSD    | Params: {count_params(vgg_net):.1f}M | Latency: {benchmark_speed(vgg_net):.2f}ms')
print(f'MobileNet-SSD| Params: {count_params(mob_net):.1f}M | Latency: {benchmark_speed(mob_net):.2f}ms')

## 6. mAP Evaluation (VOC2007 test set)

VGG16-SSD uses `eval.py` (original repo script). MobileNetV2-SSD uses `eval_mobilenet.py`,
which is identical except it loads the model with `build_mobilenet_ssd('test', ...)` and
writes results to a separate output directory so the two evals don't conflict.

In [ ]:
# VGG16 baseline mAP
!python eval.py --trained_model weights/ssd300_mAP_77.43_v2.pth --voc_root ./data/VOCdevkit

In [ ]:
# MobileNetV2-SSD mAP
!python eval_mobilenet.py --trained_model weights/mobilenet_ssd_epoch50.pth --voc_root ./data/VOCdevkit

## 7. Results Summary

| Model | mAP (VOC2007) | Params | Latency (ms) |
|---|---|---|---|
| VGG16-SSD (paper) | 77.2 | ~26M | — |
| VGG16-SSD (ours) | _fill in_ | ~26M | _measure_ |
| MobileNetV2-SSD (ours) | _fill in_ | ~4–5M | _measure_ |

**Analysis notes:**
- Where does MobileNetV2 degrade most? (Hypothesis: small objects — no 38×38 feature map)
- Speedup factor vs accuracy drop tradeoff
- Parameter reduction ratio